# 🚀 DAY 9: ENTERPRISE GEMINI AI ENGINEERING & PORTFOLIO LAB
### End-to-End Application: *The Executive Resume Bullet & Impact Optimizer*

This notebook covers:
1. **Environment Setup & Zero-Hardcoding Security** (`.env` / Colab Secrets)
2. **API Architecture, Statelessness & Role Mapping**
3. **Pre-flight Token Counting & Cost Estimation**
4. **Production Resilience (Exponential Backoff with Jitter)**
5. **Reusable Gemini Wrapper & 3-Turn Dialogue Memory**
6. **Student Lab: Structured Pydantic Output & Multi-Turn Interactive Revisions**
7. **Git Repository Hygiene**

In [ ]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & SECURE API AUTHENTICATION
# Run this cell first. Installs the Google GenAI SDK, Pydantic, and dotenv.
# ==============================================================================
!pip install -q -U google-genai pydantic python-dotenv tabulate

import os
import sys
import time
import json
import random
import getpass
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None

# Official Google GenAI SDK
from google import genai
from google.genai import types
from google.genai.errors import APIError

# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion (Zero-Hardcoding Policy)
# ------------------------------------------------------------------------------
load_dotenv()

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Model selection (defaults to gemini-3-flash-preview for high speed & quota resilience)
MODEL_NAME = os.environ.get('GEMINI_MODEL', 'gemini-3-flash-preview')

# Initialize Client
client = genai.Client(api_key=GEMINI_API_KEY)
print(f"✅ Google Gemini API Client initialized successfully! (Model: {MODEL_NAME})")

In [ ]:
# ==============================================================================
# SECTION 1: API ARCHITECTURE, STATELESSNESS & SECURITY HYGIENE
# ==============================================================================
"""
1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:
   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.
   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.
   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.

2. THE STATELESSNESS MENTAL MODEL:
   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.
   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list
     of previous (User, Model) turns and pass the cumulative array on every subsequent call.

3. MESSAGE ROLES MAPPING:
   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐
   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │
   ├──────────────────────┼─────────────────────────┼───────────────────────────┤
   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │
   │ User Message         │ role: 'user'            │ role: 'user'              │
   │ Model Response       │ role: 'assistant'       │ role: 'model'             │
   └──────────────────────┴─────────────────────────┴───────────────────────────┘
"""

In [ ]:
# ==============================================================================
# SECTION 2: PRE-FLIGHT TOKEN COUNTING & FINANCIAL COST ESTIMATION
# ==============================================================================
"""
COST ESTIMATION BEST PRACTICE:
Count input tokens BEFORE invoking expensive generation calls to protect budget thresholds.
"""
def preflight_cost_estimate(
    text_prompt: str,
    model_name: str = MODEL_NAME,
    expected_output_tokens: int = 500
) -> Dict[str, Any]:
    """Calculates exact input tokens and estimates financial cost before calling the API."""
    token_resp = client.models.count_tokens(model=model_name, contents=text_prompt)
    input_tokens = token_resp.total_tokens

    # Official Rates per 1M tokens (USD)
    pricing = {
        "gemini-3-flash-preview": {"in": 0.075, "out": 0.30},
        "gemini-flash-latest":    {"in": 0.075, "out": 0.30},
        "gemini-3.6-flash":       {"in": 0.075, "out": 0.30},
        "gemini-2.5-flash":       {"in": 0.075, "out": 0.30},
        "gemini-1.5-flash":       {"in": 0.075, "out": 0.30},
        "gemini-1.5-pro":         {"in": 1.25,  "out": 5.00}
    }
    rate = pricing.get(model_name, pricing["gemini-3-flash-preview"])

    est_cost = (input_tokens / 1e6 * rate["in"]) + (expected_output_tokens / 1e6 * rate["out"])

    return {
        "model": model_name,
        "input_tokens": input_tokens,
        "estimated_output_tokens": expected_output_tokens,
        "estimated_cost_usd": round(float(est_cost), 6),
        "cost_per_10k_calls": round(float(est_cost * 10000), 2)
    }

sample_payload = "Please summarize the last 10 quarterly financial filings of Apple, Microsoft, and Google."
estimate = preflight_cost_estimate(sample_payload, model_name=MODEL_NAME)
print("=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===")
for k, v in estimate.items():
    print(f"• {k:25s}: {v}")

In [ ]:
# ==============================================================================
# SECTION 3: PRODUCTION RESILIENCE — EXPONENTIAL BACKOFF & RETRY LOOP
# ==============================================================================
"""
HANDLING API FAILURES IN PRODUCTION:
1. Rate Limits (HTTP 429 / ResourceExhausted): Hit requests-per-minute (RPM) ceiling.
2. Transient Server Errors (HTTP 500 / 503): Temporary Google Cloud infrastructure hiccup.
3. Network Timeouts: Connection dropped during streaming.

REMEDY: EXPONENTIAL BACKOFF WITH JITTER:
Wait time = (base_delay * 2^attempt) + random_jitter
Prevents 'Thundering Herd' problem where all failed clients retry at the exact same millisecond.
"""

def execute_with_exponential_backoff(
    api_call_func,
    max_retries: int = 6,
    base_delay: float = 2.0
):
    """Wraps an API call in an exponential backoff retry loop with random jitter."""
    for attempt in range(max_retries):
        try:
            return api_call_func()
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Max retries reached. Fatal API Error: {e}")
                raise e
            code = getattr(e, 'code', getattr(e, 'status_code', type(e).__name__))
            delay = (base_delay * (2 ** attempt)) + random.uniform(0.2, 1.0)
            print(f"⚠️ Warning: Transient API Error ({code}). Retrying in {delay:.2f}s... (Attempt {attempt+1}/{max_retries})")
            time.sleep(delay)

In [ ]:
# ==============================================================================
# SECTION 4: REUSABLE GEMINI WRAPPER & 3-TURN CHAT
# ==============================================================================
def gemini_call(
    prompt: str,
    system_instruction: str = "You are a concise, helpful enterprise AI assistant.",
    temperature: float = 0.2,
    stream: bool = False,
    model: str = MODEL_NAME
) -> str:
    """Production-grade wrapper for Google Gemini API with error handling and streaming."""
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=800
    )

    if stream:
        def stream_call():
            full_text = []
            response_stream = client.models.generate_content_stream(
                model=model, contents=prompt, config=config
            )
            for chunk in response_stream:
                if chunk.text:
                    print(chunk.text, end="", flush=True)
                    full_text.append(chunk.text)
            print()
            return "".join(full_text)

        return execute_with_exponential_backoff(stream_call)
    else:
        def standard_call():
            resp = client.models.generate_content(
                model=model, contents=prompt, config=config
            )
            return resp.text.strip()

        return execute_with_exponential_backoff(standard_call)

# ------------------------------------------------------------------------------
# 3-Turn Conversational Memory Loop Demonstration
# ------------------------------------------------------------------------------
print("=== MULTI-TURN CONVERSATION LOOP ===")

conversation_history = []
system_persona = "You are a Senior PostgreSQL Database Administrator. Answer concisely in 2 sentences."

def send_chat_turn(user_message: str):
    print(f"\n👤 User: {user_message}")
    print("🤖 Assistant: ", end="")

    conversation_history.append({"role": "user", "parts": [{"text": user_message}]})

    config = types.GenerateContentConfig(
        system_instruction=system_persona,
        temperature=0.0
    )
    def call_turn():
        return client.models.generate_content(
            model=MODEL_NAME,
            contents=conversation_history,
            config=config
        )

    response = execute_with_exponential_backoff(call_turn)
    bot_reply = response.text.strip()
    print(bot_reply)
    conversation_history.append({"role": "model", "parts": [{"text": bot_reply}]})

send_chat_turn("What is the difference between a clustered and non-clustered index?")
time.sleep(1.0)
send_chat_turn("Which one is faster for range queries on primary keys?")
time.sleep(1.0)
send_chat_turn("Can a table have multiple of the faster one?")

# ==============================================================================
# SECTION 5: STUDENT LAB WORKSPACE (PORTFOLIO APPLICATION)
# ==============================================================================
### Assignment: *The Executive Resume Bullet & Impact Optimizer*

**Core Requirements:**
1. **Structured JSON Schema (Pydantic)**:
   - `original_bullet`: Raw input text
   - `xyz_formatted_bullet`: Google's XYZ Formula ("Accomplished [X], as measured by [Y], by doing [Z]")
   - `impact_metric`: Quantifiable numeric KPI
   - `action_verb`: Strong opening action verb
   - `seniority_score`: Integer rating (1 to 10) evaluating executive presence
   - `critique`: 1-sentence explanation of what was improved
2. **Interactive Revision History**: Maintain multi-turn state so users can request revisions.
3. **Schema Validation & Display**: Robust validation using Pydantic and clean presentation.
4. **Error Handling**: Wrapped in exponential backoff retry loop.

In [ ]:
# ==============================================================================
# TASK 1: DEFINE PYDANTIC SCHEMA FOR STRUCTURED RESUME OPTIMIZATION
# ==============================================================================
class ResumeBulletOptimization(BaseModel):
    """Pydantic schema enforcing structured output for executive resume bullet optimization."""
    original_bullet: str = Field(
        ...,
        description="The raw, unoptimized resume bullet provided by the user."
    )
    xyz_formatted_bullet: str = Field(
        ...,
        description="Rewritten bullet strictly following Google's XYZ Formula: 'Accomplished [X], as measured by [Y], by doing [Z]'."
    )
    impact_metric: str = Field(
        ...,
        description="The quantifiable numeric KPI (e.g., '45% latency reduction', '$1.2M in annual savings', '99.99% uptime')."
    )
    action_verb: str = Field(
        ...,
        description="Strong executive-level opening action verb in past tense (e.g., 'Spearheaded', 'Architected', 'Orchestrated', 'Transformed')."
    )
    seniority_score: int = Field(
        ...,
        ge=1,
        le=10,
        description="Integer rating (1 to 10) assessing executive presence, scope of ownership, and strategic business impact."
    )
    critique: str = Field(
        ...,
        description="A concise 1-sentence explanation of weaknesses identified in the original bullet and how this optimization elevates the candidate's profile."
    )

print("✅ Task 1: Pydantic Schema defined successfully!")

In [ ]:
# ==============================================================================
# TASK 2: BUILD THE APPLICATION ENGINE
# ==============================================================================
class ResumeOptimizerEngine:
    """
    Production-grade Engine for optimizing resume bullets using Google's XYZ formula.
    Features:
    - Structured output validation with Pydantic
    - Multi-turn conversational memory for interactive revisions
    - Production resilience via exponential backoff
    - Formatted output presentation
    """
    SYSTEM_INSTRUCTION = (
        "You are an elite Fortune 500 Executive Career Coach and Former Google Senior Tech Recruiter. "
        "Your mission is to transform weak, passive resume bullet points into high-impact, executive-tier "
        "statements using Google's proven XYZ formula: 'Accomplished [X], as measured by [Y], by doing [Z]'.\n"
        "Guidelines:\n"
        "1. Never use weak phrases like 'responsible for', 'helped with', or 'worked on'.\n"
        "2. Begin every bullet with a decisive, high-leverage action verb (e.g., Spearheaded, Architected, Engineered).\n"
        "3. Always inject plausible, rigorous quantifiable metrics (percentage gains, latency cuts, dollar savings, throughput).\n"
        "4. Provide a seniority score between 1 and 10 based on executive presence and business value.\n"
        "5. Respond strictly in valid JSON conforming to the requested schema."
    )

    def __init__(self, model_name: str = MODEL_NAME):
        self.model_name = model_name
        self.conversation_history: List[Dict[str, Any]] = []
        self.latest_result: Optional[ResumeBulletOptimization] = None

    def reset_history(self):
        """Clears the conversational context for a fresh resume bullet optimization."""
        self.conversation_history = []
        self.latest_result = None

    def _call_gemini_structured(self, contents: Any) -> ResumeBulletOptimization:
        """Executes API call configured with structured JSON schema and exponential backoff."""
        # TODO 2.1: Configure GenerateContentConfig with temperature=0.1, response_mime_type='application/json', and response_schema
        config = types.GenerateContentConfig(
            system_instruction=self.SYSTEM_INSTRUCTION,
            temperature=0.1,
            response_mime_type="application/json",
            response_schema=ResumeBulletOptimization
        )

        # TODO 2.2: Execute API call with exponential backoff
        def api_invocation():
            resp = client.models.generate_content(
                model=self.model_name,
                contents=contents,
                config=config
            )
            return resp

        response = execute_with_exponential_backoff(api_invocation)
        parsed_data = ResumeBulletOptimization.model_validate_json(response.text)
        return parsed_data

    def optimize_bullet(self, raw_bullet: str) -> ResumeBulletOptimization:
        """Optimizes a raw bullet and initiates the conversational revision history."""
        self.reset_history()
        user_prompt = f"Optimize this resume bullet into Google's XYZ formula:\n\"{raw_bullet}\""

        self.conversation_history.append({
            "role": "user",
            "parts": [{"text": user_prompt}]
        })

        result = self._call_gemini_structured(self.conversation_history)

        self.conversation_history.append({
            "role": "model",
            "parts": [{"text": result.model_dump_json()}]
        })

        self.latest_result = result
        return result

    def request_revision(self, feedback: str) -> ResumeBulletOptimization:
        """Multi-turn revision: refines previous output based on user feedback."""
        if not self.conversation_history:
            raise ValueError("No previous bullet found in history. Call optimize_bullet() first.")

        revision_prompt = f"Revise the previous bullet with this specific feedback:\n\"{feedback}\""

        self.conversation_history.append({
            "role": "user",
            "parts": [{"text": revision_prompt}]
        })

        result = self._call_gemini_structured(self.conversation_history)

        self.conversation_history.append({
            "role": "model",
            "parts": [{"text": result.model_dump_json()}]
        })

        self.latest_result = result
        return result

    @staticmethod
    def display_optimization_card(opt: ResumeBulletOptimization, title: str = "RESUME BULLET OPTIMIZATION"):
        """Formats and displays the structured optimization beautifully in the console."""
        border = "=" * 78
        score_bar = '★' * opt.seniority_score + '☆' * (10 - opt.seniority_score)
        print(f"\n{border}")
        print(f"🎯 {title.upper()}")
        print(border)
        print(f"📌 ORIGINAL BULLET : {opt.original_bullet}")
        print(f"🚀 XYZ OPTIMIZED   : {opt.xyz_formatted_bullet}")
        print(f"⚡ ACTION VERB     : {opt.action_verb}")
        print(f"📊 IMPACT KPI      : {opt.impact_metric}")
        print(f"⭐ SENIORITY SCORE : {opt.seniority_score}/10 [{score_bar}]")
        print(f"💡 CRITIQUE        : {opt.critique}")
        print(f"{border}\n")

print("✅ Task 2: ResumeOptimizerEngine built successfully!")

In [ ]:
# ==============================================================================
# TASK 3: TEST APPLICATION ON REAL-WORLD WEAK BULLETS
# ==============================================================================
engine = ResumeOptimizerEngine(model_name=MODEL_NAME)

weak_bullets = [
    "Helped with the database and made it run faster.",
    "Wrote code for a customer login page in React.",
    "Responsible for managing a team of sales reps and tracking leads in CRM."
]

print("=== TESTING REAL-WORLD WEAK BULLETS ===")
for idx, bullet in enumerate(weak_bullets, 1):
    print(f"\n>>> Test Case #{idx}: Input Bullet: '{bullet}'")
    result = engine.optimize_bullet(bullet)
    engine.display_optimization_card(result, title=f"Test Case #{idx} Results")
    time.sleep(1.5)

# ------------------------------------------------------------------------------
# DEMONSTRATE INTERACTIVE REVISION HISTORY (MULTI-TURN)
# ------------------------------------------------------------------------------
print("\n=== DEMONSTRATING MULTI-TURN INTERACTIVE REVISION HISTORY ===")
print("1. Base optimization on: 'Helped with the database and made it run faster.'")
base_result = engine.optimize_bullet("Helped with the database and made it run faster.")
engine.display_optimization_card(base_result, title="Turn 1: Base Optimization")
time.sleep(1.5)

print("\n2. Revision Turn 1: 'Revise for a Principal Architect role. Emphasize multi-region PostgreSQL replication, $250K cloud cost reduction, and zero downtime.'")
revised_1 = engine.request_revision(
    "Revise for a Principal Architect role. Emphasize multi-region PostgreSQL replication, $250K cloud cost reduction, and zero downtime."
)
engine.display_optimization_card(revised_1, title="Turn 2: Multi-Region & Cost Focus")
time.sleep(1.5)

print("\n3. Revision Turn 2: 'Push the seniority score to 10/10 with an elite Fortune 50 C-suite perspective.'")
revised_2 = engine.request_revision(
    "Push the seniority score to 10/10 with an elite Fortune 50 C-suite perspective."
)
engine.display_optimization_card(revised_2, title="Turn 3: Fortune 50 Executive Polish")

print("✅ Task 3: All real-world test cases & multi-turn revisions completed!")

In [ ]:
# ==============================================================================
# SECTION 6: GIT REPOSITORY HYGIENE — CREATING .ENV AND .GITIGNORE
# ==============================================================================
"""
INSTRUCTIONS FOR PUSHING TO GITHUB SAFELY:

1. Create a `.env` file locally:
   GEMINI_API_KEY=your_actual_key_here

2. Create a `.gitignore` file in your project root containing:
   .env
   .env.local
   *.joblib
   __pycache__/
   .ipynb_checkpoints/

3. In your Python script (`app.py`), load the key cleanly via:
   from dotenv import load_dotenv
   load_dotenv()
   api_key = os.getenv("GEMINI_API_KEY")
"""

# Script to generate .gitignore locally in Colab / local folder
with open(".gitignore", "w") as f:
    f.write(".env\n.env.*\n*.joblib\n__pycache__/\n.ipynb_checkpoints/\n")

print("✅ '.gitignore' template created successfully!")